In [8]:
import mammoth
from bs4 import BeautifulSoup
import zipfile, os, shutil

# ===== User configuration =====
word_file = "FIRE SAFETY AND PROTECTION NARIND VERMA.docx"
project_name = "FIRE_SAFETY_AND_PROTECTION_NARIND_VERMA"
author_name = "Narind Verma"
affiliation = "B.Tech CSE (AIML), UPES, Dehradun, India"
email = "as.softech@gmail.com"
title = "FIRE SAFETY AND PROTECTION USING AIML"
# ==============================

os.makedirs(project_name, exist_ok=True)
img_dir = os.path.join(project_name, "images")
os.makedirs(img_dir, exist_ok=True)

print("🔄 Converting Word → HTML (for structured parsing)...")
with open(word_file, "rb") as docx_file:
    result = mammoth.convert_to_html(docx_file, convert_image=mammoth.images.img_element(lambda image: {
        "src": os.path.join("images", image.content_type.split("/")[-1] + "_" + str(abs(hash(image.alt_text or 'img'))) + ".png")
    }))
    html = result.value

# Parse HTML and replace tags with LaTeX equivalents
print("🧠 Processing document structure...")
soup = BeautifulSoup(html, "html.parser")

def replace_tags(soup):
    for b in soup.find_all(["strong", "b"]):
        b.string = f"\\textbf{{{b.get_text()}}}"
    for i in soup.find_all(["em", "i"]):
        i.string = f"\\textit{{{i.get_text()}}}"
    for h1 in soup.find_all("h1"):
        h1.string = f"\\section*{{{h1.get_text()}}}"
    for h2 in soup.find_all("h2"):
        h2.string = f"\\subsection*{{{h2.get_text()}}}"
    for h3 in soup.find_all("h3"):
        h3.string = f"\\subsubsection*{{{h3.get_text()}}}"
replace_tags(soup)

paragraphs = []
for p in soup.find_all("p"):
    text = p.get_text().strip()
    if text:
        paragraphs.append(text + "\n\n")

latex_body = "".join(paragraphs)

# Springer Nature LaTeX template
latex_template = f"""
\\documentclass[pdflatex,sn-mathphys-num]{{sn-jnl}}
\\usepackage{{graphicx}}
\\usepackage{{amsmath,amssymb,amsfonts}}
\\usepackage{{hyperref}}
\\begin{{document}}

\\title[{title}]{{{title}}}

\\author*[1]{{\\fnm{{{author_name}}}}}
\\affil[1]{{\\orgdiv{{Department of Computer Science}}, \\orgname{{UPES}}, \\city{{Dehradun}}, \\country{{India}}}}
\\email{{{email}}}

\\maketitle

{latex_body}

\\bibliography{{sn-bibliography}}

\\end{{document}}
"""

tex_path = os.path.join(project_name, "main.tex")
with open(tex_path, "w", encoding="utf-8") as f:
    f.write(latex_template)

# Bundle ZIP
zip_name = f"{project_name}.zip"
print("📦 Creating Overleaf ZIP project...")
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(project_name):
        for file in files:
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, project_name)
            zipf.write(full_path, rel_path)

print(f"✅ Done! Upload '{zip_name}' to Overleaf and compile with sn-jnl.cls.")


🔄 Converting Word → HTML (for structured parsing)...
🧠 Processing document structure...
📦 Creating Overleaf ZIP project...
✅ Done! Upload 'FIRE_SAFETY_AND_PROTECTION_NARIND_VERMA.zip' to Overleaf and compile with sn-jnl.cls.
